# Chapter 5

Add your content here.

In [1]:
import numpy as np
import random

# --- 1. Setup the Environment ---
TARGET = 10
START = 0
MAX_STEPS = 5

class State:
    def __init__(self, value, history):
        self.value = value
        self.history = history # The path taken

    def __repr__(self):
        return f"Val: {self.value}, Path: {self.history}"

# Actions: +1 or -1
actions = [1, -1]

# --- 2. Define the Value Function (The Oracle) ---
# In Deep Learning, this is a Neural Network. 
# Here, we simply define V(s) as how close we are to the target.
# Let's normalize it to be between 0 and 1 roughly.
def V(current_number):
    distance = abs(TARGET - current_number)
    # A simple heuristic: The closer to target, the higher the V.
    # If distance is 0, V is 1. If distance is 20, V is 0.
    return max(0, 1 - (distance / 20))

# --- 3. Outcome Reward Model (ORM) Search ---
# This agent wanders randomly. It only knows if it won at the very end.
def run_orm_episode():
    current_state = State(START, [])
    
    for _ in range(MAX_STEPS):
        action = random.choice(actions) # Random walk
        new_val = current_state.value + action
        current_state = State(new_val, current_state.history + [action])
    
    # Sparse Signal: Only check at the very end
    if current_state.value == TARGET:
        return 1, current_state # Success
    else:
        return 0, current_state # Failure (0 info about how close we were)

# --- 4. Process Advantage Verifier (PAV) Search ---
# This agent looks at V(s_next) - V(s) to make decisions.
def run_pav_episode():
    current_state = State(START, [])
    
    print(f"Start: {current_state}")
    
    for _ in range(MAX_STEPS):
        best_action = None
        best_advantage = -float('inf')
        
        # Look ahead at all possible actions (Tree Search)
        for action in actions:
            # 1. Simulate next state s_{t+1}
            next_val = current_state.value + action
            
            # 2. Calculate Value V(s) and V(s_{t+1})
            v_current = V(current_state.value)
            v_next = V(next_val)
            
            # 3. Calculate Advantage: r = V(s') - V(s)
            advantage = v_next - v_current
            
            # Greedy choice: pick action with highest advantage
            if advantage > best_advantage:
                best_advantage = advantage
                best_action = action
        
        # Take the best step
        new_val = current_state.value + best_action
        current_state = State(new_val, current_state.history + [best_action])
        print(f"Step taken: {best_action}. Advantage: {best_advantage:.2f}. New State: {current_state.value}")

    return current_state

# --- 5. Execution ---
print("--- Running ORM (Random/Sparse) ---")
score, final_state = run_orm_episode()
print(f"ORM Final State: {final_state.value}. Success: {score}")
print("\nNote: ORM likely failed and has no idea why.\n")

print("--- Running PAV (Process Supervision) ---")
final_pav_state = run_pav_episode()
print(f"PAV Final State: {final_pav_state.value}")

--- Running ORM (Random/Sparse) ---
ORM Final State: -1. Success: 0

Note: ORM likely failed and has no idea why.

--- Running PAV (Process Supervision) ---
Start: Val: 0, Path: []
Step taken: 1. Advantage: 0.05. New State: 1
Step taken: 1. Advantage: 0.05. New State: 2
Step taken: 1. Advantage: 0.05. New State: 3
Step taken: 1. Advantage: 0.05. New State: 4
Step taken: 1. Advantage: 0.05. New State: 5
PAV Final State: 5
